In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Circuit Analysis Code Evaluation

This notebook evaluates the code implementing the circuit analysis from `/net/scratch2/smallyan/filter_eval`.

## Evaluation Criteria
For each code block:
1. **Runnable (Y/N)** - Executes without error
2. **Correct-Implementation (Y/N)** - Logic implements described computation correctly
3. **Redundant (Y/N)** - Duplicates another block
4. **Irrelevant (Y/N)** - Does not contribute to project goal

In [2]:
# Check GPU availability
import torch
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = "cuda"
else:
    device = "cpu"
print(f"Using device: {device}")

CUDA available: True
CUDA device: NVIDIA H200 NVL
CUDA memory: 150.1 GB
Using device: cuda


In [3]:
# Initialize evaluation tracking
evaluation_results = []
corrected_blocks = 0
total_failed_blocks = 0

import sys
sys.path.insert(0, "/net/scratch2/smallyan/filter_eval")
os.chdir("/net/scratch2/smallyan/filter_eval")
print(f"Changed to: {os.getcwd()}")

Changed to: /net/scratch2/smallyan/filter_eval


## Evaluating demo.ipynb

### Cell 1: Autoreload setup

In [4]:
# Cell 1: Autoreload setup
cell_id = "demo.ipynb - Cell 1 (autoreload)"
try:
    %load_ext autoreload
    %autoreload 2
    runnable = "Y"
    error_note = ""
except Exception as e:
    runnable = "N"
    error_note = str(e)
    total_failed_blocks += 1

evaluation_results.append({
    "cell_id": cell_id,
    "runnable": runnable,
    "correct_implementation": "Y",  # Standard IPython autoreload
    "redundant": "N",
    "irrelevant": "N",  # Useful for development
    "error_note": error_note
})
print(f"{cell_id}: Runnable={runnable}")

demo.ipynb - Cell 1 (autoreload): Runnable=Y


### Cell 2: Model Loading

In [5]:
# Cell 2: Model loading
cell_id = "demo.ipynb - Cell 2 (model loading)"
try:
    import torch
    import transformers
    from src.models import ModelandTokenizer

    print(f"{torch.__version__=}, {torch.version.cuda=}")
    print(f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}")
    print(f"{transformers.__version__=}")

    model_key = "meta-llama/Llama-3.3-70B-Instruct"
    
    mt = ModelandTokenizer(
        model_key=model_key,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )
    runnable = "Y"
    error_note = ""
except Exception as e:
    runnable = "N"
    error_note = str(e)
    total_failed_blocks += 1

print(f"\n{cell_id}: Runnable={runnable}")

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


meta-llama/Llama-3.3-70B-Instruct not found in /net/projects/chai-lab/shared_models
If not found in cache, model will be downloaded from HuggingFace to cache directory


`torch_dtype` is deprecated! Use `dtype` instead!


torch.__version__='2.7.1+cu118', torch.version.cuda='11.8'
torch.cuda.is_available()=True, torch.cuda.device_count()=1, torch.cuda.get_device_name()='NVIDIA H200 NVL'
transformers.__version__='4.57.3'


tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [6]:
# Check if model finished loading
print(f"Model loaded: {mt is not None}")
print(f"Model name: {mt.name}")
print(f"Number of layers: {mt.n_layer}")
print(f"Device: {mt.device}")